# Лабораторная работа №1 — Регрессия: Wine Quality

**Цель:** пройти полный цикл регрессии: EDA → корректное разбиение → константный baseline → линейная регрессия / случайный лес / градиентный бустинг → сравнение RMSE и MAE → интерпретация → ошибки.

**Условие преподавателя:** RMSE на скрытой тестовой выборке должен быть строго меньше **0.70**. Локальный test — только приближённая проверка, он не равен скрытому leaderboard.

**Данные:** оригинальные `winequality-red.csv` и `winequality-white.csv` (UCI Wine Quality, Cortez et al.). Объединяем красные и белые вина и явно добавляем признак `is_red`; ТЗ не ограничивает тип вина. Оба CSV сохранены в `regression/data/`.

**Принцип честного эксперимента:** тест отделяется до исследования зависимостей; модель и её параметры выбираются исключительно по кросс-валидации на train. Тест открывается один раз в заключительной части. Преобразования обучаются только на тренировочных фолдах с помощью Pipeline.

Источник: https://archive.ics.uci.edu/dataset/186/wine+quality

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)


## 1. Загрузка и понимание данных

Физико-химические показатели вина (`alcohol`, кислотность, плотность и др.) выступают признаками, `quality` — числовая оценка качества. Хотя исходные оценки дискретны, по требованию лабораторной решаем задачу **регрессии**.

In [ ]:
def find_data_dir():
    candidates = [Path("data"), Path("regression/data"), Path("../data")]
    for folder in candidates:
        if (folder / "winequality-red.csv").exists() and (folder / "winequality-white.csv").exists():
            return folder
    raise FileNotFoundError("Не найдены CSV: запустите ноутбук из корня репозитория или папки regression.")

data_dir = find_data_dir()
red = pd.read_csv(data_dir / "winequality-red.csv", sep=";")
white = pd.read_csv(data_dir / "winequality-white.csv", sep=";")
red["is_red"] = 1
white["is_red"] = 0
data = pd.concat([red, white], ignore_index=True)
assert data.shape[0] == 6497 and "quality" in data
print("Красное:", len(red), "| Белое:", len(white), "| Всего:", len(data))
display(data.head())
display(data.describe().T)


## 2. Защищённое разбиение train/test

Выбираем **80% train / 20% test**, воспроизводимый `random_state=42`. Стратификация по `quality` нужна, чтобы редкие оценки были представлены в обеих частях. Тест **не участвует** в EDA, выборе признаков, гиперпараметров и победителя. Многократная проверка гипотез по одному тесту приводит к косвенному переобучению на него. Повторяющиеся наблюдения могут искусственно улучшать оценку — ниже изучим их число (без изменения заранее зафиксированного протокола).

In [ ]:
X = data.drop(columns="quality")
y = data["quality"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
assert set(X_train.index).isdisjoint(set(X_test.index))
print("Дубликатов признаков во всём датасете:", X.duplicated().sum())
print("Дубликатов train-признаков:", X_train.duplicated().sum())
print("Важно: наличие одинаковых признаков в разных строках может завышать качество случайного разбиения.")


## 3. EDA — только на обучающей выборке

Смотрим на баланс целевых оценок и тип вина, пропуски, корреляции. Корреляции описывают связь, **не причинность**; почти нулевая корреляция не означает, что признак бесполезен для нелинейной модели.

In [ ]:
train_view = X_train.assign(quality=y_train)
print("Пропуски (train):")
display(train_view.isna().sum().sort_values(ascending=False).head(12))
print("Распределение quality (train):")
display(y_train.value_counts().sort_index().rename("count").to_frame())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=y_train, ax=axes[0], color="steelblue")
axes[0].set_title("Оценки качества (train)")
sns.boxplot(data=train_view, x="is_red", y="quality", ax=axes[1])
axes[1].set_title("Quality: белое (0) и красное (1)")
plt.tight_layout()
plt.show()


**Интерпретация:** наиболее частые значения quality можно предсказывать проще; экстремальные оценки представлены небольшим числом примеров, поэтому ошибки на них могут быть выше. Разница распределений по цвету — повод оставить `is_red` среди признаков, но это не доказательство причинного влияния цвета.

In [ ]:
corr = train_view.corr(numeric_only=True)["quality"].drop("quality").sort_values()
display(corr.rename("pearson_corr").to_frame())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
corr.plot.barh(ax=axes[0], title="Корреляция Пирсона с quality")
sns.scatterplot(data=train_view, x="alcohol", y="quality", hue="is_red", alpha=.16, ax=axes[1], legend="brief")
axes[1].set_title("Alcohol и quality (train)")
plt.tight_layout()
plt.show()


**Вывод по EDA:** признаки с наиболее сильной по модулю корреляцией — кандидаты на важные линейные предикторы. Важно проверить, будут ли они столь же значимыми в лесу и бустинге: эти методы учитывают нелинейности и взаимодействия. Корреляции не используются для удаления признаков «на глаз», чтобы избежать необоснованной селекции.

## 4. Метрики и константный baseline

**MAE** — средняя абсолютная ошибка (в баллах качества); **RMSE** — корень из средней квадратичной ошибки. Возведение в квадрат делает RMSE более чувствительной к большим промахам. Поэтому MAE и RMSE для той же модели могут различаться, если есть редкие очень большие ошибки.

Для минимизации **MSE и RMSE** оптимальная константа — **среднее арифметическое y на обучении**, не медиана. На CV среднее вычисляется на каждом тренировочном фолде через `DummyRegressor(strategy="mean")`.

In [ ]:
def metrics(y_true, y_pred):
    return {"RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAE": mean_absolute_error(y_true, y_pred)}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"}
baseline = DummyRegressor(strategy="mean")
cv_base = cross_validate(baseline, X_train, y_train, cv=cv, scoring=scoring)
print("Baseline: CV RMSE =", round(-cv_base["test_rmse"].mean(), 4),
      "| CV MAE =", round(-cv_base["test_mae"].mean(), 4))


## 5. Обязательные модели: Linear Regression, Random Forest и Gradient Boosting

Линейная модель использует `StandardScaler` для корректного сравнения коэффициентов в стандартных отклонениях; импутация обучается внутри Pipeline. Лес и бустинг не требуют масштабирования. Для выбора параметров используется **только** 5-fold CV на train; все сравниваются по одной и той же основной метрике **RMSE** и с одинаковыми фолдами.

In [ ]:
linear = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
forest = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])
boosting = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
])

experiments = {
    "Linear Regression": (linear, {}),
    "Random Forest": (forest, {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 12],
        "model__min_samples_leaf": [1, 3],
    }),
    "Gradient Boosting": (boosting, {
        "model__n_estimators": [150, 300],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3],
    })
}


In [ ]:
from sklearn.base import clone

cv_rows = [{
    "Модель": "Baseline",
    "CV RMSE": -cv_base["test_rmse"].mean(),
    "CV MAE": -cv_base["test_mae"].mean()
}]
fitted_models = {"Baseline": clone(baseline).fit(X_train, y_train)}
best_params = {}

for name, (estimator, grid) in experiments.items():
    if grid:
        search = GridSearchCV(estimator, grid, scoring="neg_root_mean_squared_error",
                              cv=cv, n_jobs=-1, refit=True)
        search.fit(X_train, y_train)
        selected = search.best_estimator_
        best_params[name] = search.best_params_
    else:
        selected = clone(estimator).fit(X_train, y_train)
        best_params[name] = "без настраиваемых гиперпараметров"
    # Честная CV-оценка выбранного значения гиперпараметров;
    # для подбора сетки это слегка оптимистичная оценка, так как выбирали по тем же фолдам.
    scores = cross_validate(selected, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({"Модель": name, "CV RMSE": -scores["test_rmse"].mean(),
                    "CV MAE": -scores["test_mae"].mean()})
    fitted_models[name] = selected
    print(name, "| параметры:", best_params[name])

cv_table = pd.DataFrame(cv_rows).sort_values("CV RMSE").reset_index(drop=True)
display(cv_table.round(4))
best_name = cv_table.iloc[0]["Модель"]
print("Выбрана по CV:", best_name)


**Важно:** при подборе гиперпараметров максимизация по той же CV-сетке может делать CV-результат слегка оптимистичным. Для учебного сравнения этого достаточно; в строгом исследовании применяют nested cross-validation. Финальный test ниже не участвует в выборе.

## 6. Коэффициенты линейной модели и importance деревьев

После `StandardScaler` абсолютные коэффициенты линейной модели сопоставимы по масштабу: они описывают изменение предсказания при росте признака на одно стандартное отклонение **при прочих равных**. Однако коррелированные признаки и масштабирование бинарного `is_red` осложняют содержательную интерпретацию. В деревьях impurity-based importance может смещаться в пользу непрерывных признаков; feature importance не устанавливает причинность.

In [ ]:
features = X_train.columns.tolist()
coef = pd.Series(fitted_models["Linear Regression"].named_steps["model"].coef_, index=features)
importance_rf = pd.Series(fitted_models["Random Forest"].named_steps["model"].feature_importances_, index=features)
importance_gb = pd.Series(fitted_models["Gradient Boosting"].named_steps["model"].feature_importances_, index=features)
display(pd.DataFrame({
    "Linear |standardized coef|": coef.abs().sort_values(ascending=False),
    "Forest importance": importance_rf,
    "Boosting importance": importance_gb
}).fillna(0).sort_values("Forest importance", ascending=False).round(3))
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
coef.sort_values().plot.barh(ax=axs[0], title="Linear: стандарт. коэффициенты")
importance_rf.sort_values().plot.barh(ax=axs[1], title="Random Forest: importance")
importance_gb.sort_values().plot.barh(ax=axs[2], title="Gradient Boosting: importance")
plt.tight_layout()
plt.show()
print("Топ-5 Forest:", importance_rf.nlargest(5).index.tolist())
print("Топ-5 Boosting:", importance_gb.nlargest(5).index.tolist())
print("Пересечение топ-5:", sorted(set(importance_rf.nlargest(5).index) & set(importance_gb.nlargest(5).index)))


**Обсуждение:** сопоставьте топ-признаки графиков с корреляциями из EDA. Различия ожидаемы: линейная регрессия описывает аддитивные линейные эффекты, деревья могут учитывать пороговые зависимости и сочетания. Важность признака **не означает**, что его изменение вызовет изменение качества: возможны корреляции с невидимыми факторами и взаимодействия.

## 7. Финальная тестовая оценка — один раз, после выбора по CV

Все модели оцениваются на **том же** `X_test`; ни одна не дообучается на нём и параметры не меняются на основе таблицы. Таблица обязательно включает baseline, RMSE и MAE. **Победителя выбирали по CV**, а не по тестовой метрике.

In [ ]:
test_rows = []
predictions = {}
for name in ["Baseline", "Linear Regression", "Random Forest", "Gradient Boosting"]:
    pred = fitted_models[name].predict(X_test)
    predictions[name] = pred
    test_rows.append({"Модель": name, **metrics(y_test, pred)})
test_table = pd.DataFrame(test_rows).set_index("Модель").loc[
    ["Baseline", "Linear Regression", "Random Forest", "Gradient Boosting"]
]
display(test_table.round(4))
print("Порог скрытого теста RMSE < 0.70: локальная проверка НЕ подтверждает прохождение leaderboard.")
print("Выбранная по CV модель:", best_name)
print("Её локальный test RMSE:", round(test_table.loc[best_name, "RMSE"], 4))
print("Превзошли константу (test):", test_table.index[test_table["RMSE"] < test_table.loc["Baseline", "RMSE"]].tolist())


## 8. Анализ ошибок выбранной по CV модели

Рассмотрим остатки `y_true - y_pred`, наиболее крупные ошибки, средние ошибки по значениям `quality` и различия между редкими/частыми оценками. Редкие классы часто сложнее: модель обучается в основном на типичных оценках, а RMSE не гарантирует равного качества на всех подгруппах.

In [ ]:
best_pred = predictions[best_name]
errors = pd.DataFrame({"actual": y_test, "predicted": best_pred}, index=y_test.index)
errors["residual"] = errors["actual"] - errors["predicted"]
errors["absolute_error"] = errors["residual"].abs()

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(data=errors, x="residual", bins=35, kde=True, ax=axes[0])
axes[0].axvline(0, color="black", linestyle="--")
axes[0].set_title("Распределение остатков")
sns.scatterplot(data=errors, x="actual", y="predicted", alpha=.35, ax=axes[1])
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], "k--", linewidth=1)
axes[1].set_title("Истинные vs предсказанные")
errors.groupby("actual")["absolute_error"].mean().plot.bar(ax=axes[2], title="MAE по исходному quality")
plt.tight_layout()
plt.show()

print("Топ-10 крупных ошибок:")
display(errors.nlargest(10, "absolute_error").round(3))
print("Ошибки по оценкам quality:")
display(errors.groupby("actual").agg(
    n=("absolute_error", "size"),
    MAE=("absolute_error", "mean"),
    mean_prediction=("predicted", "mean")
).round(3))


In [ ]:
print("Объекты с крупнейшими ошибками (только анализ, без перенастройки):")
worst = errors.nlargest(5, "absolute_error")
display(X_test.loc[worst.index].join(worst).round(3))
print("Количество train-примеров каждой оценки:")
display(y_train.value_counts().sort_index().rename("train_count").to_frame())


## 9. Итоговые выводы

1. Сравните значения в `cv_table` и `test_table`: какие модели действительно лучше константного baseline, насколько и почему?
2. Укажите модель, **выбранную по train CV** (`best_name`), и её тестовые MAE/RMSE. Не подбирайте модель заново по test.
3. Сопоставьте важные признаки линейной регрессии, леса, бустинга и корреляционный анализ. Помните: importance не равен причинному эффекту.
4. По таблице ошибок опишите, какие редкие оценки `quality` предсказываются хуже всего и приведите 2–3 конкретных примера.
5. Назовите ограничения: одинаковые химические признаки у повторов, субъективность экспертной оценки качества, ненаблюдаемые факторы, несовпадение локального и скрытого теста.

**Для защиты:** объясните разницу MAE/RMSE, почему baseline для RMSE — среднее, зачем random_state, зачем CV и Pipeline, почему нельзя подбирать гиперпараметры по тесту, что такое переобучение, и почему важность признаков не доказывает причинность.